# OpenCV 5 Object Detection Use Cases: Traffic Counting, Parking Monitoring and Safety Zones

In the first two parts of this series we ran **YOLO26** with nothing but the new **OpenCV 5 DNN engine** on a CPU,
first in [Python](https://learnopencv.com/how-to-run-object-detection-with-opencv-5/) and then in
[C++](https://learnopencv.com/opencv-5-cpp-object-detection-yolo26/). A detector on its own only draws boxes.
This notebook turns those boxes into answers people actually ask for:

1. **Traffic counting**: how many cars, buses, trucks and motorbikes crossed a line, in which direction, in which lane
2. **Parking monitoring**: how many cars are parked in each area, which ones are still moving, even when the camera drifts
3. **Safety zones**: who stepped into a restricted area, for how long, with a snapshot and an event log

Everything runs on the CPU with **OpenCV 5 + NumPy only**. Ultralytics is used once, to export the ONNX file.
The tracker, the line counter, the zone logic and the camera-drift compensation are all written here with plain OpenCV,
so you can read every line and port it to C++ later.

**Examples in this notebook (all free-license footage: 9 clips and 8 photos):**

| # | Use case | Scene | What it shows |
|---|---|---|---|
| 1 | Traffic | Istanbul highway (drone, slow zoom) | two-way counting, buses and trucks, lines pinned to the road |
| 2 | Traffic | Kathmandu street | dense motorbikes, rider merging, pedestrians on the sidewalk |
| 3 | Traffic | Suburban divided road | per-lane counts |
| 4 | Parking | Supermarket car park (drone) | parked vs moving cars per area, drift compensation |
| 1b, 2b | Traffic | Highway photo, Chennai street photo | vehicles per lane; vehicle mix with tiling and rider merging |
| 4b | Parking | Still photo | single pass vs tiled detection for distant cars |
| 4c | Parking | Straight-down view | where a COCO detector stops working |
| 4d, 4e | Parking | Dusk lot, near-overhead drone photo | low light; tiling rescues tiny cars |
| 5b, 6b | Zones | Platform photo, warehouse photo | people behind the yellow line; workers in a forklift aisle |
| 5 | Zones | Rail platform | platform-edge alert with dwell time |
| 6 | Zones | Factory floor | machine-area alert with snapshots and CSV log |
| 7 | Zones | Warehouse | a moving safety bubble around a forklift |

> **Runtime:** the OpenCV 5 pip wheel is CPU-only, so a GPU does not speed up inference here. A free Colab CPU or T4 runtime
> is enough. Every example has a `max_seconds` knob if you want a faster first run.

## 1. Setup

Colab ships with OpenCV 4.x (three different packages of it). We install Ultralytics first (only for the ONNX export),
then remove every OpenCV 4 build and install the OpenCV 5 wheel, so exactly one `cv2` is left.

In [ ]:
!pip install -q ultralytics onnx onnxslim
!pip uninstall -y -q opencv-python opencv-python-headless opencv-contrib-python opencv-contrib-python-headless
!pip install -q opencv-python==5.0.0.93

In [ ]:
import os, time, json, csv, glob, subprocess, base64, urllib.request
from collections import deque
from dataclasses import dataclass, field

import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, Image, display

print("OpenCV", cv2.__version__)
print("CPU threads OpenCV will use:", cv2.getNumThreads(), "| logical CPUs:", os.cpu_count())
assert cv2.__version__.startswith("5."), "Restart the runtime (Runtime > Restart session) and run this cell again."
os.makedirs("models", exist_ok=True); os.makedirs("outputs", exist_ok=True)

## 2. Export YOLO26 to ONNX

We export the **nano** model at two input sizes. 640 is fast and fine when objects are reasonably large.
960 costs about 2.5x more per frame but finds far more small objects, such as distant motorbikes. We use whichever
each scene needs, and the benchmark at the end shows the trade-off.

The export settings match Part 1: opset 12, a fixed input size, and `nms=False`. YOLO26 is NMS-free by design, so
the output is a fixed `[1, 300, 6]` tensor of final detections.

In [ ]:
from ultralytics import YOLO
import shutil

for size in (640, 960):
    path = YOLO("yolo26n.pt").export(format="onnx", opset=12, imgsz=size, dynamic=False, nms=False, simplify=True)
    shutil.move(path, f"models/yolo26n_{size}.onnx")
print(os.listdir("models"))

## 3. Get the footage from GitHub

All clips and photos are free to use: the Pexels videos and photos are under the [Pexels license](https://www.pexels.com/license/),
and the factory clip is Intel's [sample-videos](https://github.com/intel-iot-devkit/sample-videos) repository (CC BY 4.0).
They are provided here for educational purposes only, not for resale. All 10 clips and 8 photos are hosted in the
LearnOpenCV GitHub repository and downloaded into a local `assets` folder, so the notebook runs the same way on Colab or on your own machine.

In [ ]:
# one place to change if the repository moves
GITHUB_ASSETS = "https://raw.githubusercontent.com/spmallick/learnopencv/master/opencv5-yolo26-use-cases/assets/"
ASSETS = "assets"                      # relative to the working directory, so the same code runs anywhere
CLIPS = ["traffic_highway_istanbul.mp4", "traffic_street_kathmandu.mp4", "traffic_suburban_lanes.mp4",
         "parking_supermarket.mp4", "parking_rooftop.mp4", "parking_rows_clip.mp4", "parking_dusk_clip.mp4",
         "rail_platform.mp4", "warehouse_forklift.mp4", "worker_zone.mp4"]
PHOTOS = ["highway_lanes.jpg", "street_mixed_traffic.jpg", "parking_rows_still.jpg", "parking_dusk_still.jpg",
          "parking_topdown_still.jpg", "parking_aerial_angled.jpg", "platform_yellow_line.jpg", "warehouse_aisle.jpg"]

os.makedirs(ASSETS, exist_ok=True)
for name in CLIPS + PHOTOS:
    dst = os.path.join(ASSETS, name)
    if not os.path.exists(dst):
        urllib.request.urlretrieve(GITHUB_ASSETS + name, dst)
    assert os.path.getsize(dst) > 1024, f"{name} did not download correctly"
for name in CLIPS:
    cap = cv2.VideoCapture(os.path.join(ASSETS, name))
    print(f"{name:32s} {int(cap.get(3))}x{int(cap.get(4))} @ {cap.get(5):.0f} fps, {cap.get(7) / cap.get(5):5.1f} s")
print(len(PHOTOS), "photos:", ", ".join(PHOTOS))

In [ ]:
def show_frames(path, times, cols=2, width=16):
    """Show frames of a video at the given times (seconds) in a grid."""
    cap = cv2.VideoCapture(path); fps = cap.get(cv2.CAP_PROP_FPS)
    last = max(int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) - 2, 0)
    imgs = []
    for t in times:
        k = min(int(t * fps), last)                   # clamp, e.g. when MAX_SECONDS shortened the clip
        cap.set(cv2.CAP_PROP_POS_FRAMES, k); ok, f = cap.read()
        if ok: imgs.append((k / fps, cv2.cvtColor(f, cv2.COLOR_BGR2RGB)))
    if not imgs:
        print("no frames to show"); return
    rows = (len(imgs) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(width, width * 0.29 * rows), squeeze=False)
    for ax in axes.ravel(): ax.axis("off")
    for ax, (t, im) in zip(axes.ravel(), imgs):
        ax.imshow(im); ax.set_title(f"t = {t:.1f} s", fontsize=10)
    plt.tight_layout(); plt.show()


def to_h264(src, dst=None, width=1280, max_seconds=None, crf=26):
    """OpenCV writes mp4v; browsers (and WordPress) want H.264. ffmpeg converts it."""
    dst = dst or src.replace(".mp4", "_h264.mp4")
    cmd = ["ffmpeg", "-y", "-loglevel", "error", "-i", src]
    if max_seconds: cmd += ["-t", str(max_seconds)]
    cmd += ["-vf", f"scale={width}:-2", "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", str(crf),
            "-preset", "veryfast", "-movflags", "+faststart", dst]
    subprocess.run(cmd, check=True)
    return dst


def show_video(src, width=900, max_seconds=12):
    """Embed a short H.264 preview of a result video in the notebook."""
    p = to_h264(src, src.replace(".mp4", "_preview.mp4"), width=width, max_seconds=max_seconds, crf=30)
    b64 = base64.b64encode(open(p, "rb").read()).decode()
    display(HTML(f'<video width="{width}" controls loop autoplay muted playsinline>'
                 f'<source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))


show_frames("assets/traffic_highway_istanbul.mp4", [1, 8], cols=2, width=14)

## 4. The building blocks

Three use cases, one small toolbox. Every piece below is plain OpenCV and NumPy.

### 4.1 The detector: YOLO26 on OpenCV 5

`cv2.dnn.readNet(model, "", "", engine)` is the OpenCV 5 signature: the fourth argument picks the inference engine.
`ENGINE_NEW` is the rewritten CPU engine. Because YOLO26 is NMS-free, post-processing is just a confidence filter
and undoing the letterbox.

`tiled()` is for still images with tiny objects: it runs the model on overlapping tiles so each object covers more
input pixels, then merges duplicates. It is the only place where we need NMS, because neighboring tiles see the same
car twice.

In [ ]:
COCO = [
    "person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck", "boat",
    "traffic light", "fire hydrant", "stop sign", "parking meter", "bench", "bird", "cat",
    "dog", "horse", "sheep", "cow", "elephant", "bear", "zebra", "giraffe", "backpack",
    "umbrella", "handbag", "tie", "suitcase", "frisbee", "skis", "snowboard", "sports ball",
    "kite", "baseball bat", "baseball glove", "skateboard", "surfboard", "tennis racket",
    "bottle", "wine glass", "cup", "fork", "knife", "spoon", "bowl", "banana", "apple",
    "sandwich", "orange", "broccoli", "carrot", "hot dog", "pizza", "donut", "cake", "chair",
    "couch", "potted plant", "bed", "dining table", "toilet", "tv", "laptop", "mouse",
    "remote", "keyboard", "cell phone", "microwave", "oven", "toaster", "sink",
    "refrigerator", "book", "clock", "vase", "scissors", "teddy bear", "hair drier",
    "toothbrush",
]


ENGINES = {
    "auto": cv2.dnn.ENGINE_AUTO,        # new engine, falls back to classic if a layer is unsupported
    "new": cv2.dnn.ENGINE_NEW,          # the rewritten OpenCV 5 engine (CPU)
    "classic": cv2.dnn.ENGINE_CLASSIC,  # the 4.x engine (needed for the CUDA backend)
}


class Detector:
    """YOLO26 on OpenCV 5 DNN.

    YOLO26 is NMS-free: the ONNX output is a fixed [1, 300, 6] tensor where every
    row is already a final detection (x1, y1, x2, y2, score, class_id) in the
    640x640 letterboxed input space. We only filter by score and undo the
    letterbox, no transpose and no cv2.dnn.NMSBoxes.
    """

    def __init__(self, onnx_path, imgsz=640, engine="new", conf=0.30, classes=None):
        self.net = cv2.dnn.readNet(onnx_path, "", "", ENGINES[engine])
        self.imgsz = imgsz
        self.conf = conf
        # keep only these class names (None = keep all 80)
        self.keep = None if classes is None else {COCO.index(c) for c in classes}

    def _letterbox(self, img):
        h, w = img.shape[:2]
        s = self.imgsz / max(h, w)
        nh, nw = round(h * s), round(w * s)
        top, left = (self.imgsz - nh) // 2, (self.imgsz - nw) // 2
        canvas = np.full((self.imgsz, self.imgsz, 3), 114, np.uint8)
        canvas[top:top + nh, left:left + nw] = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
        return canvas, s, left, top

    def __call__(self, img, conf=None):
        """Return an (N, 6) float array: x1, y1, x2, y2, score, class_id in image pixels."""
        conf = self.conf if conf is None else conf
        canvas, s, left, top = self._letterbox(img)
        blob = cv2.dnn.blobFromImage(canvas, 1 / 255.0, swapRB=True)
        self.net.setInput(blob)
        out = self.net.forward()[0]                      # (300, 6)
        out = out[out[:, 4] >= conf]
        if self.keep is not None:
            out = out[np.isin(out[:, 5].astype(int), list(self.keep))]
        if len(out) == 0:
            return np.zeros((0, 6), np.float32)
        out = out.copy()
        out[:, [0, 2]] = (out[:, [0, 2]] - left) / s     # undo letterbox
        out[:, [1, 3]] = (out[:, [1, 3]] - top) / s
        h, w = img.shape[:2]
        out[:, [0, 2]] = out[:, [0, 2]].clip(0, w - 1)
        out[:, [1, 3]] = out[:, [1, 3]].clip(0, h - 1)
        return out

    def tiled(self, img, tiles=(2, 2), overlap=0.2, conf=None, iou=0.5):
        """Detect on overlapping tiles and merge, for small objects in large frames.

        Each tile is run through the network at full model resolution, so a car
        that is 15 px wide in the whole frame becomes ~30 px wide in its tile.
        Duplicates from overlapping tiles are merged with class-aware NMS
        (the only place we need NMS, because tiles see the same object twice).
        """
        h, w = img.shape[:2]
        ty, tx = tiles
        th, tw = int(h / (ty - (ty - 1) * overlap)), int(w / (tx - (tx - 1) * overlap))
        ys = np.linspace(0, h - th, ty).astype(int)
        xs = np.linspace(0, w - tw, tx).astype(int)
        dets = []
        for y0 in ys:
            for x0 in xs:
                d = self(img[y0:y0 + th, x0:x0 + tw], conf)
                d[:, [0, 2]] += x0
                d[:, [1, 3]] += y0
                dets.append(d)
        dets.append(self(img, conf))                    # the full frame catches large objects
        d = np.concatenate(dets)
        if len(d) == 0:
            return d
        boxes = [[float(a), float(b), float(c - a), float(e - b)] for a, b, c, e in d[:, :4]]
        keep = cv2.dnn.NMSBoxesBatched(boxes, d[:, 4].tolist(), d[:, 5].astype(int).tolist(), conf or self.conf, iou)
        return d[np.array(keep, dtype=int).reshape(-1)]

### 4.2 A tracker that gives every object an ID

Counting needs identity: "car #17 crossed the line" can happen only once, while "a car is near the line" could be
true for twenty frames. `IoUTracker` is a greedy tracker in about sixty lines:

* predict where each track should be using a smoothed velocity
* **pass 1:** match predictions to detections by IoU, best first, same class only
* **pass 2:** match what is left by center distance, measured in object sizes. Small or fast objects (a motorbike
  far away) often have zero IoU with their prediction, and this pass is what keeps their IDs stable
* start new tracks, let missed tracks coast for a few frames, then drop them

A track is reported only after `min_hits` matches, which filters one-frame false positives.

In [ ]:
def iou_matrix(a, b):
    """Pairwise IoU between boxes a (N,4) and b (M,4) in x1,y1,x2,y2."""
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), np.float32)
    x1 = np.maximum(a[:, None, 0], b[None, :, 0])
    y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2])
    y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-6)


@dataclass
class Track:
    tid: int
    box: np.ndarray                 # x1, y1, x2, y2
    cls: int
    score: float
    hits: int = 1
    missed: int = 0
    trail: list = field(default_factory=list)
    vel: np.ndarray = field(default_factory=lambda: np.zeros(4, np.float32))

    @property
    def center(self):
        return np.array([(self.box[0] + self.box[2]) / 2, (self.box[1] + self.box[3]) / 2])

    @property
    def foot(self):
        """Bottom-center point: where a person or vehicle touches the ground."""
        return np.array([(self.box[0] + self.box[2]) / 2, self.box[3]])


class IoUTracker:
    """Greedy IoU tracker with constant-velocity prediction.

    Every frame we (1) predict where each track should be, (2) match predictions
    to new detections by IoU, highest first, (3) update matched tracks, start new
    tracks for unmatched detections, and drop tracks that have been missing for
    `max_missed` frames. It is tiny, has no dependencies, and is good enough for
    a fixed camera at normal frame rates.
    """

    def __init__(self, iou_thr=0.3, max_missed=15, min_hits=3, trail_len=40, dist_thr=0.8):
        self.iou_thr, self.max_missed, self.min_hits, self.trail_len = iou_thr, max_missed, min_hits, trail_len
        self.dist_thr = dist_thr                     # max center distance in units of object size (pass 2)
        self.tracks, self.next_id = [], 1

    def _assign(self, t, det):
        new = det[:4]
        steps = t.missed + 1                           # frames since the last real observation
        t.vel = 0.6 * t.vel + 0.4 * (new - (t.box - t.vel * t.missed)) / steps   # smoothed per-frame velocity
        t.box, t.score = new.copy(), float(det[4])
        t.hits, t.missed = t.hits + 1, 0

    def update(self, dets):
        preds = np.array([t.box + t.vel for t in self.tracks]).reshape(-1, 4)
        ious = iou_matrix(preds, dets[:, :4])
        matched_t, matched_d = set(), set()
        # pass 1: greedy matching, best IoU first, same class only
        for ti, di in zip(*np.unravel_index(np.argsort(-ious, axis=None), ious.shape)):
            if ious[ti, di] < self.iou_thr:
                break
            if ti in matched_t or di in matched_d or self.tracks[ti].cls != int(dets[di, 5]):
                continue
            self._assign(self.tracks[ti], dets[di])
            matched_t.add(ti); matched_d.add(di)
        # pass 2: small or fast objects often have zero IoU with their prediction,
        # so match what is left by center distance, relative to the object's size
        rest_t = [i for i in range(len(self.tracks)) if i not in matched_t]
        rest_d = [i for i in range(len(dets)) if i not in matched_d]
        if rest_t and rest_d:
            pc = (preds[rest_t, :2] + preds[rest_t, 2:]) / 2
            dc = (dets[rest_d, :2] + dets[rest_d, 2:4]) / 2
            size = np.maximum(preds[rest_t, 2] - preds[rest_t, 0], preds[rest_t, 3] - preds[rest_t, 1])
            dist = np.linalg.norm(pc[:, None] - dc[None], axis=2) / (size[:, None] + 1e-6)
            for a, b in zip(*np.unravel_index(np.argsort(dist, axis=None), dist.shape)):
                if dist[a, b] > self.dist_thr:
                    break
                ti, di = rest_t[a], rest_d[b]
                if ti in matched_t or di in matched_d or self.tracks[ti].cls != int(dets[di, 5]):
                    continue
                self._assign(self.tracks[ti], dets[di])
                matched_t.add(ti); matched_d.add(di)
        for ti, t in enumerate(self.tracks):
            if ti not in matched_t:
                t.missed += 1
                t.box = t.box + t.vel                   # coast on the prediction
        for di in range(len(dets)):
            if di not in matched_d:
                self.tracks.append(Track(self.next_id, dets[di, :4].copy(), int(dets[di, 5]), float(dets[di, 4])))
                self.next_id += 1
        self.tracks = [t for t in self.tracks if t.missed <= self.max_missed]
        for t in self.tracks:
            if t.missed == 0:
                t.trail.append(tuple(t.center.astype(int)))
                t.trail = t.trail[-self.trail_len:]
        return [t for t in self.tracks if t.hits >= self.min_hits and t.missed == 0]

### 4.3 Keeping lines and zones pinned when the camera moves

Lines and zones are drawn once, on the first frame. That works for a bolted-down CCTV camera, but two of our clips are
drone shots that drift and zoom. `CameraAligner` matches ORB features between the current frame and a keyframe, fits a
transform with RANSAC, and chains it back to the first frame. Every point we test (a car's position) is mapped into the
first frame, and every line or zone we draw is mapped into the current frame.

Two details matter in practice:

* **Mask out the detections** before extracting features. Moving cars are full of corners, and without the mask they drag
  the fit along with the traffic.
* **Pick the motion model for the scene.** A parking lot is a plane, so a full homography is exact. A highway with
  buildings behind it has depth, and a full homography over-fits; a 4-DOF similarity (shift, rotation, zoom) is stable.

In [ ]:
class CameraAligner:
    """Keeps a transform from the current frame back to a reference frame.

    Zones and lines are drawn once on the reference frame. If the camera drifts
    (a drone, a pole swaying in the wind, a slow zoom), we match ORB features
    between the current frame and a keyframe, fit a transform with RANSAC, and
    chain it to the keyframe's own transform.

    Two details make this robust:
      * detected objects are masked out before extracting features, so only the
        static background votes (moving cars would otherwise drag the fit along)
      * `motion` picks the model: "homography" (8 DOF) for a flat scene such as a
        parking lot, "similarity" (4 DOF: shift, rotation, zoom) for scenes with
        depth such as a highway, where a full homography over-fits.
    Features are matched on a half-size copy; the result is rescaled to full
    resolution with H_full = S^-1 @ H_small @ S.
    """

    def __init__(self, ref_frame, motion="homography", n_features=1500, rekey_every=20, min_inliers=30,
                 scale=0.5, boxes=None):
        self.motion, self.scale = motion, scale
        self.S = np.diag([scale, scale, 1.0])
        self.Si = np.linalg.inv(self.S)
        self.orb = cv2.ORB_create(n_features)
        self.matcher = cv2.BFMatcher(cv2.NORM_HAMMING)
        self.rekey_every, self.min_inliers = rekey_every, min_inliers
        self._set_key(ref_frame, np.eye(3), boxes)
        self.H = np.eye(3)                       # current frame -> reference
        self.inliers, self.n = 0, 0

    def _features(self, frame, boxes):
        g = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        g = cv2.resize(g, None, fx=self.scale, fy=self.scale, interpolation=cv2.INTER_AREA)
        mask = np.full(g.shape, 255, np.uint8)
        if boxes is not None:
            for x1, y1, x2, y2 in (np.asarray(boxes)[:, :4] * self.scale).astype(int):
                mask[max(y1, 0):y2, max(x1, 0):x2] = 0          # ignore pixels on detected objects
        return self.orb.detectAndCompute(g, mask)

    def _set_key(self, frame, H_key_to_ref, boxes=None):
        self.kp, self.des = self._features(frame, boxes)
        self.H_key = H_key_to_ref

    def update(self, frame, boxes=None):
        self.n += 1
        kp, des = self._features(frame, boxes)
        if des is not None and self.des is not None and len(des) > 8 and len(self.des) > 8:
            pairs = self.matcher.knnMatch(des, self.des, k=2)
            good = [m for m, n in (p for p in pairs if len(p) == 2) if m.distance < 0.75 * n.distance]
            if len(good) >= 8:
                src = np.float32([kp[m.queryIdx].pt for m in good])
                dst = np.float32([self.kp[m.trainIdx].pt for m in good])
                if self.motion == "similarity":
                    A, mask = cv2.estimateAffinePartial2D(src, dst, method=cv2.RANSAC, ransacReprojThreshold=2.0)
                    Hs = None if A is None else np.vstack([A, [0, 0, 1]])
                else:
                    Hs, mask = cv2.findHomography(src, dst, cv2.RANSAC, 2.0)
                self.inliers = int(mask.sum()) if mask is not None else 0
                if Hs is not None and self.inliers >= self.min_inliers:
                    self.H = self.H_key @ (self.Si @ Hs @ self.S)
        if self.n % self.rekey_every == 0:
            self._set_key(frame, self.H, boxes)   # new keyframe keeps the overlap large
        return self.H


def to_ref(H, pts):
    return cv2.perspectiveTransform(np.float32(pts).reshape(-1, 1, 2), H).reshape(-1, 2)

### 4.4 Lines and zones

A line crossing is a sign change of a cross product. `LineCounter` remembers which side of the line each track was on and
counts a crossing when the side flips, once per track. It also insists that the object stayed on the old side for at
least two updates, which removes jitter from boxes that sit right on the line.

A zone is a polygon; `cv2.pointPolygonTest` tells us whether a point is inside. For people and vehicles we test the
**foot point** (bottom-center of the box), because that is where they stand. All coordinates in the configs are normalized
(0 to 1), so they work at any resolution.

In [ ]:
def side_of_line(p, a, b):
    """Sign of the cross product: >0 on one side of line a->b, <0 on the other."""
    return np.sign((b[0] - a[0]) * (p[1] - a[1]) - (b[1] - a[1]) * (p[0] - a[0]))


class LineCounter:
    """Counts tracks whose anchor point crosses segment a->b, in both directions."""

    def __init__(self, a, b, name="line", anchor="center", classes=None):
        self.a, self.b, self.name, self.anchor = np.array(a, float), np.array(b, float), name, anchor
        self.classes = None if classes is None else {COCO.index(c) for c in classes}
        self.last_side, self.counted = {}, {}
        self.count_in, self.count_out = 0, 0
        self.by_class = {}
        self.flash = 0

    def _within_segment(self, p):
        ab = self.b - self.a
        t = np.dot(p - self.a, ab) / (np.dot(ab, ab) + 1e-9)
        return -0.05 <= t <= 1.05

    def update(self, tracks, H=None):
        """H (optional) maps current-frame points into the frame the line was drawn on."""
        for t in tracks:
            if self.classes is not None and t.cls not in self.classes:
                continue
            p = t.foot if self.anchor == "foot" else t.center
            if H is not None:
                p = to_ref(H, [p])[0]
            s = side_of_line(p, self.a, self.b)
            prev, streak = self.last_side.get(t.tid, (None, 0))
            self.last_side[t.tid] = (s, streak + 1 if s == prev else 1)
            # a crossing must come from a side the object stayed on for at least 2 updates,
            # which filters out box jitter right on the line and tracks that start on top of it
            if prev is None or s == 0 or prev == 0 or s == prev or streak < 2 or t.tid in self.counted:
                continue
            if not self._within_segment(p):
                continue
            direction = "in" if s > 0 else "out"
            self.counted[t.tid] = direction
            if direction == "in":
                self.count_in += 1
            else:
                self.count_out += 1
            name = COCO[t.cls]
            self.by_class.setdefault(name, [0, 0])[0 if direction == "in" else 1] += 1
            self.flash = 6


class Zone:
    """A named polygon. Membership is tested with cv2.pointPolygonTest."""

    def __init__(self, pts, name="zone", color=(0, 200, 255)):
        self.pts = np.array(pts, np.int32)
        self.name, self.color = name, color

    def contains(self, p):
        return cv2.pointPolygonTest(self.pts, (float(p[0]), float(p[1])), False) >= 0


def scaled(pts, w, h):
    """Polygons are defined in normalized 0..1 coordinates so they work at any resolution."""
    return [(int(x * w), int(y * h)) for x, y in pts]

### 4.5 Drawing and the video loop

Drawing helpers scale line widths and fonts with the frame height, so labels stay readable from 480p to 4K.
`VideoJob` reads, processes and writes a video, and keeps the pipeline FPS (detection + tracking + logic + drawing,
without decoding and encoding). `every=2` processes every second frame of 50/60 fps clips, which is plenty for counting.

In [ ]:
PALETTE = [(255, 99, 71), (60, 180, 75), (0, 130, 200), (245, 130, 48), (145, 30, 180),
           (70, 240, 240), (240, 50, 230), (210, 245, 60), (250, 190, 212), (0, 128, 128)]


def ui_scale(img):
    """A size factor so lines and text stay readable from 480p to 4K."""
    return max(img.shape[0] / 720.0, 0.6)


def color_for(i):
    return PALETTE[i % len(PALETTE)]


def draw_label(img, text, org, color, scale=1.0, text_color=(255, 255, 255)):
    fs, th = 0.55 * scale, max(1, int(round(1.5 * scale)))
    (tw, tht), base = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, fs, th)
    x, y = int(org[0]), int(org[1])
    y = max(y, tht + base + 4)
    x = max(0, min(x, img.shape[1] - tw - 6))          # keep the label inside the frame
    cv2.rectangle(img, (x, y - tht - base - 4), (x + tw + 6, y), color, -1)
    cv2.putText(img, text, (x + 3, y - base - 1), cv2.FONT_HERSHEY_SIMPLEX, fs, text_color, th, cv2.LINE_AA)


def draw_box(img, box, color, label=None, scale=1.0, thickness=None):
    x1, y1, x2, y2 = [int(v) for v in box]
    cv2.rectangle(img, (x1, y1), (x2, y2), color, thickness or max(2, int(2 * scale)), cv2.LINE_AA)
    if label:
        draw_label(img, label, (x1, y1), color, scale)


def draw_trail(img, trail, color, scale=1.0):
    for i in range(1, len(trail)):
        t = max(1, int(scale * 3 * i / len(trail)))
        cv2.line(img, trail[i - 1], trail[i], color, t, cv2.LINE_AA)


def draw_zone(img, zone, alpha=0.25, color=None, label=True, scale=1.0, label_pos="top"):
    """label_pos "top" puts the name at the polygon's top-left; "bottom" at its lowest edge, which
    reads better for lane quads that taper towards the horizon."""
    color = color or zone.color
    overlay = img.copy()
    cv2.fillPoly(overlay, [zone.pts], color)
    cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0, img)
    cv2.polylines(img, [zone.pts], True, color, max(2, int(2 * scale)), cv2.LINE_AA)
    if label:
        if label_pos == "bottom":
            ymax = zone.pts[:, 1].max()
            low = zone.pts[zone.pts[:, 1] >= ymax - 0.05 * img.shape[0]]
            x, y = int(low[:, 0].mean()) - int(40 * scale), min(int(ymax) - int(8 * scale), img.shape[0] - int(8 * scale))
        else:
            x, y = zone.pts[:, 0].min(), zone.pts[:, 1].min()
        draw_label(img, zone.name, (max(x, 0), y), color, scale)


def draw_panel(img, lines, org=(12, 12), scale=1.0, bg=(20, 20, 20), alpha=0.65, width=None):
    """A translucent info panel with one text line per entry. Entries can be (text, color)."""
    fs, th = 0.62 * scale, max(1, int(round(1.6 * scale)))
    lh = int(30 * scale)
    items = [(l, (255, 255, 255)) if isinstance(l, str) else l for l in lines]
    w = width or max(cv2.getTextSize(t, cv2.FONT_HERSHEY_SIMPLEX, fs, th)[0][0] for t, _ in items) + int(24 * scale)
    h = lh * len(items) + int(14 * scale)
    x, y = org
    overlay = img.copy()
    cv2.rectangle(overlay, (x, y), (x + w, y + h), bg, -1)
    cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0, img)
    for i, (t, c) in enumerate(items):
        cv2.putText(img, t, (x + int(12 * scale), y + int(8 * scale) + lh * (i + 1) - int(8 * scale)),
                    cv2.FONT_HERSHEY_SIMPLEX, fs, c, th, cv2.LINE_AA)
    return w, h


class VideoJob:
    """Read a video, call process(frame, idx) -> frame, and write the result.

    Keeps a running FPS for the whole pipeline (decode + detect + logic + draw)
    and can stop early with max_seconds so demos stay quick on CPU.
    """

    def __init__(self, src, dst, max_seconds=None, out_width=None, every=1):
        self.cap = cv2.VideoCapture(src)
        if not self.cap.isOpened():
            raise IOError(f"cannot open {src}")
        self.fps_in = self.cap.get(cv2.CAP_PROP_FPS) or 30
        self.every = every                              # process every n-th frame (for 50/60 fps clips)
        self.w = int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        self.h = int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        self.out_w = out_width or self.w
        self.out_h = int(round(self.h * self.out_w / self.w / 2) * 2)
        n = int(self.cap.get(cv2.CAP_PROP_FRAME_COUNT))
        self.max_frames = n if not max_seconds else min(n if n > 0 else 10**9, int(max_seconds * self.fps_in))
        self.writer = cv2.VideoWriter(dst, cv2.VideoWriter_fourcc(*"mp4v"), self.fps_in / every, (self.out_w, self.out_h))
        self.dst = dst
        self.proc_times = []

    def run(self, process, log_every=100):
        idx, t_start = 0, time.perf_counter()
        while idx < self.max_frames:
            ok, frame = self.cap.read()
            if not ok:
                break
            if idx % self.every == 0:
                if frame.shape[1] != self.out_w:
                    frame = cv2.resize(frame, (self.out_w, self.out_h), interpolation=cv2.INTER_AREA)
                t0 = time.perf_counter()
                frame = process(frame, idx)
                self.proc_times.append(time.perf_counter() - t0)
                self.writer.write(frame)
            idx += 1
            if log_every and idx % log_every == 0:
                print(f"  frame {idx}/{self.max_frames}  {self.fps:.1f} FPS")
        self.cap.release()
        self.writer.release()
        print(f"done: {len(self.proc_times)} frames, {self.fps:.1f} FPS average, total {time.perf_counter() - t_start:.1f}s -> {self.dst}")
        return self

    @property
    def fps(self):
        return len(self.proc_times) / max(sum(self.proc_times), 1e-6)

## 5. Use case 1: Traffic counting

`count_traffic` wires the pieces together: detect, optionally align, track, update every counting line, log each
crossing to a CSV, and draw. Each line can have its own class filter (a bus-lane line only counts buses) and its own
direction labels.

On busy streets COCO labels a motorbike and its rider separately. `merge_riders` drops a `person` box when that person
overlaps a two-wheeler and their feet end on it, so riders are not drawn twice or counted as pedestrians.

In [ ]:
GREEN, RED, AMBER, WHITE, CYAN = (80, 200, 60), (40, 40, 230), (0, 190, 255), (255, 255, 255), (230, 200, 40)
CLASS_COLORS = {"car": (255, 140, 40), "truck": (40, 170, 255), "bus": (200, 80, 220),
                "motorcycle": (60, 220, 120), "bicycle": (200, 220, 60), "person": (90, 90, 250)}


def class_color(cls_id):
    return CLASS_COLORS.get(COCO[cls_id], color_for(cls_id))


def count_traffic(cfg, detector=None):
    """Count objects crossing one or more lines.

    cfg keys
        video, out             input and output paths
        model, imgsz, conf     detector settings (ignored if `detector` is passed)
        classes                list of COCO names to keep, e.g. ["car", "truck", "bus"]
        lines                  list of dicts {name, a, b, labels=("in","out"), anchor="center"|"foot"}
        max_seconds, every     clip length and frame stride
        align                  True if the camera drifts: lines stay pinned to the road (ORB + homography)
        merge_riders           True to fold a rider's "person" box into the motorcycle/bicycle under it
        csv                    optional path for a per-crossing event log
    """
    det = detector or Detector(cfg["model"], cfg.get("imgsz", 640), conf=cfg.get("conf", 0.3), classes=cfg["classes"])
    tracker = IoUTracker(iou_thr=cfg.get("iou", 0.25), max_missed=cfg.get("max_missed", 12), min_hits=3)
    job = VideoJob(cfg["video"], cfg["out"], cfg.get("max_seconds"), cfg.get("out_width"), cfg.get("every", 1))
    W, H = job.out_w, job.out_h
    counters = [LineCounter(scaled([l["a"]], W, H)[0], scaled([l["b"]], W, H)[0], l["name"], l.get("anchor", "center"),
                            l.get("classes")) for l in cfg["lines"]]
    labels = [l.get("labels", ("in", "out")) for l in cfg["lines"]]
    events = []
    s = ui_scale(np.zeros((H, W), np.uint8))
    state = {"aligner": None, "riders": 0}

    def process(frame, idx):
        dets = det(frame)
        Hc = None
        if cfg.get("align"):
            if state["aligner"] is None:
                state["aligner"] = CameraAligner(frame, motion=cfg.get("align_motion", "similarity"), boxes=dets)
            Hc = state["aligner"].update(frame, dets)
        if cfg.get("merge_riders"):
            dets = merge_riders(dets)
        tracks = tracker.update(dets)
        before = [(c.count_in, c.count_out) for c in counters]
        for c in counters:
            c.update(tracks, Hc)
        # log new crossings
        for c, (bi, bo), lab in zip(counters, before, labels):
            if (c.count_in, c.count_out) != (bi, bo):
                for tid, d in list(c.counted.items())[-((c.count_in - bi) + (c.count_out - bo)):]:
                    t = next((t for t in tracker.tracks if t.tid == tid), None)
                    events.append({"time_s": round(idx / job.fps_in, 2), "line": c.name, "track_id": tid,
                                   "class": COCO[t.cls] if t else "?", "direction": lab[0] if d == "in" else lab[1]})
        # draw
        for t in tracks:
            col = class_color(t.cls)
            counted = any(t.tid in c.counted for c in counters)
            draw_trail(frame, t.trail, col, s)
            draw_box(frame, t.box, col, f"{COCO[t.cls]} #{t.tid}", s * 0.85, thickness=max(2, int(s * (3 if counted else 2))))
        for c, lab in zip(counters, labels):
            col = AMBER if c.flash > 0 else WHITE
            c.flash = max(0, c.flash - 1)
            ends = np.array([c.a, c.b]) if Hc is None else to_ref(np.linalg.inv(Hc), [c.a, c.b])
            a, b = tuple(ends[0].astype(int)), tuple(ends[1].astype(int))
            cv2.line(frame, a, b, (0, 0, 0), max(4, int(6 * s)), cv2.LINE_AA)
            cv2.line(frame, a, b, col, max(2, int(3 * s)), cv2.LINE_AA)
            draw_label(frame, f"{c.name}: {lab[0]} {c.count_in} | {lab[1]} {c.count_out}",
                       (min(a[0], b[0]), min(a[1], b[1]) - 6), (40, 40, 40), s * 0.9)
        # totals per class across all lines
        totals = {}
        for c in counters:
            for k, (i, o) in c.by_class.items():
                totals[k] = totals.get(k, 0) + i + o
        rows = [("TRAFFIC COUNT", AMBER)]
        rows += [(f"{k:<11s}{v:>4d}", class_color(COCO.index(k))) for k, v in sorted(totals.items(), key=lambda kv: -kv[1])]
        rows += [f"total      {sum(totals.values()):>4d}", (f"on screen  {len(tracks):>4d}", (200, 200, 200)),
                 (f"{job.fps:5.1f} FPS  OpenCV 5", (160, 160, 160))]
        draw_panel(frame, rows, (int(12 * s), int(12 * s)), s * 0.9)
        return frame

    job.run(process)
    if cfg.get("csv"):
        _write_csv(cfg["csv"], events, ["time_s", "line", "track_id", "class", "direction"])
    summary = {c.name: {"in": c.count_in, "out": c.count_out, "by_class": c.by_class} for c in counters}
    return {"summary": summary, "events": events, "fps": job.fps, "frames": len(job.proc_times)}


def merge_riders(dets):
    """Drop "person" boxes that are riders sitting on a detected motorcycle or bicycle.

    COCO labels the rider and the bike separately. On a busy street that doubles
    the clutter and, worse, a rider can be counted as a pedestrian. A person is a
    rider when they overlap a two-wheeler horizontally and their feet end inside
    the bike's box.
    """
    bikes = dets[np.isin(dets[:, 5], [COCO.index("motorcycle"), COCO.index("bicycle")])]
    if len(bikes) == 0:
        return dets
    keep = []
    for d in dets:
        if int(d[5]) != COCO.index("person"):
            keep.append(True)
            continue
        pw = d[2] - d[0]
        ov = np.clip(np.minimum(d[2], bikes[:, 2]) - np.maximum(d[0], bikes[:, 0]), 0, None) / max(pw, 1)
        feet_in = (d[3] > bikes[:, 1]) & (d[3] < bikes[:, 3] + 0.15 * (bikes[:, 3] - bikes[:, 1]))
        keep.append(not np.any((ov > 0.5) & feet_in))
    return dets[np.array(keep)]


def _write_csv(path, rows, fields):
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        w.writerows(rows)

In [ ]:
M = lambda f: f"assets/{f}"
O = lambda f: os.path.join("outputs", f)
VEH = ["car", "truck", "bus", "motorcycle"]
MAX_SECONDS = None          # set to e.g. 8 for a quick first run
RESULTS = {}


def run_example(name, fn, cfg):
    cfg = dict(cfg)
    cfg.pop("kind", None)
    cfg["model"] = os.path.join("models", cfg["model"])
    if MAX_SECONDS:
        cfg["max_seconds"] = min(MAX_SECONDS, cfg.get("max_seconds") or MAX_SECONDS)
    res = fn(cfg)
    RESULTS[name] = {"fps": round(res["fps"], 2), "frames": res["frames"], "model": os.path.basename(cfg["model"])}
    return res

### Example 1: A two-way highway in Istanbul

Three lines: one per carriageway, plus one across the Metrobus lane that only counts buses. The drone slowly pulls
back during the clip, so `align=True` keeps the lines on the asphalt. The 640 model is enough because the lines sit in
the near field, where vehicles are large.

In [ ]:
cfg = dict(
    kind="traffic", video=M("traffic_highway_istanbul.mp4"), out=O("01_traffic_highway.mp4"),
    csv=O("01_traffic_highway.csv"),
    model="yolo26n_640.onnx", imgsz=640, conf=0.3, classes=VEH, every=1, align=True,
    lines=[dict(name="Main road", a=(0.24, 0.62), b=(0.665, 0.62), labels=("towards", "away"),
                classes=["car", "truck", "bus", "motorcycle"]),
           dict(name="Side road", a=(0.71, 0.55), b=(1.0, 0.55), labels=("towards", "away"),
                classes=["car", "truck", "bus", "motorcycle"]),
           dict(name="Metrobus", a=(0.30, 0.25), b=(0.33, 0.50), labels=("east", "west"), classes=["bus"])],
)
res = run_example("1 highway", count_traffic, cfg)
print(json.dumps(res["summary"], indent=1))

In [ ]:
show_video(O("01_traffic_highway.mp4"))
show_frames(O("01_traffic_highway.mp4"), [4, 14])

### Example 2: Motorbikes and pedestrians in Kathmandu

A dense, mixed street. Here the 960 model matters: on the same frames it finds roughly 2.5x more motorbikes than the 640
model. Vehicles are counted on the road line; pedestrians are counted on a separate sidewalk line using their foot point.

In [ ]:
cfg = dict(
    kind="traffic", video=M("traffic_street_kathmandu.mp4"), out=O("02_traffic_street.mp4"),
    csv=O("02_traffic_street.csv"),
    model="yolo26n_960.onnx", imgsz=960, conf=0.3, classes=["car", "truck", "bus", "motorcycle", "bicycle", "person"],
    every=2, merge_riders=True,
    lines=[dict(name="Road", a=(0.12, 0.62), b=(0.66, 0.62), labels=("towards", "away"),
                classes=["car", "truck", "bus", "motorcycle", "bicycle"]),
           dict(name="Sidewalk", a=(0.66, 0.60), b=(0.99, 0.60), labels=("towards", "away"), anchor="foot",
                classes=["person"])],
)
res = run_example("2 street", count_traffic, cfg)
print(json.dumps(res["summary"], indent=1))

In [ ]:
show_video(O("02_traffic_street.mp4"))
show_frames(O("02_traffic_street.mp4"), [5, 12])

### Example 3: Per-lane counts on a divided road

Lane-level counts are just more lines: one short segment per lane. Lanes 1 and 2 carry traffic towards the camera,
lanes 3 and 4 carry it away.

In [ ]:
cfg = dict(
    kind="traffic", video=M("traffic_suburban_lanes.mp4"), out=O("03_traffic_lanes.mp4"), csv=O("03_traffic_lanes.csv"),
    model="yolo26n_640.onnx", imgsz=640, conf=0.3, classes=VEH, every=1,
    lines=[dict(name="Lane 1", a=(0.10, 0.72), b=(0.24, 0.72), labels=("towards", "away")),
           dict(name="Lane 2", a=(0.24, 0.72), b=(0.375, 0.72), labels=("towards", "away")),
           dict(name="Lane 3", a=(0.44, 0.72), b=(0.61, 0.72), labels=("towards", "away")),
           dict(name="Lane 4", a=(0.61, 0.72), b=(0.78, 0.72), labels=("towards", "away"))],
)
res = run_example("3 lanes", count_traffic, cfg)
print(json.dumps(res["summary"], indent=1))

In [ ]:
show_video(O("03_traffic_lanes.mp4"))
import pandas as pd
pd.read_csv(O("03_traffic_lanes.csv")).head(10)

### Examples 1b and 2b: Counting on photos

A photo cannot give you crossings, but it can give you a density snapshot: vehicles per lane, or the vehicle mix on a
street. `count_image` takes the same normalized polygons, here six lane quads on a highway photo and one road polygon on
a dense street in Chennai, and assigns every detection to the polygon that contains its ground point. On the street photo
tiling roughly triples the count, and `merge_riders` keeps the people on motorbikes out of the pedestrian total.

In [ ]:
def anchor_point(det, anchor="footprint"):
    """Where an object touches the ground: foot = bottom center; footprint = a little above it."""
    x1, y1, x2, y2 = det[:4]
    return ((x1 + x2) / 2, y2) if anchor == "foot" else ((x1 + x2) / 2, y2 - (y2 - y1) * 0.25)


def count_image(img_path, out_path, areas, detector, tiles=None, merge=False, anchor="footprint",
                title="COUNT (still image)", color=GREEN, label_pos="top"):
    """Count objects per area and per class on a single photo.

    Works for lanes on a highway photo, blocks of a parking lot, or a whole road.
    `tiles` switches on tiled inference for small objects; `merge` folds riders
    into their motorbikes. Returns ({area: {class: n}}, outside, dets).
    """
    img = cv2.imread(img_path)
    H, W = img.shape[:2]
    s = ui_scale(img)
    dets = detector.tiled(img, tiles) if tiles else detector(img)
    if merge:
        dets = merge_riders(dets)
    zones = [Zone(scaled(a["pts"], W, H), a["name"]) for a in areas]
    counts, outside = {a["name"]: {} for a in areas}, 0
    for d in dets:
        p = anchor_point(d, anchor)
        z = next((z for z in zones if z.contains(p)), None)
        if z is None:
            outside += 1
            continue
        name = COCO[int(d[5])]
        counts[z.name][name] = counts[z.name].get(name, 0) + 1
    for z in zones:
        n = sum(counts[z.name].values())
        draw_zone(img, Zone(z.pts, f"{z.name}: {n}", color), alpha=0.15, scale=s * 0.9, label_pos=label_pos)
    for d in dets:
        draw_box(img, d[:4], class_color(int(d[5])), None, s * 0.7, thickness=max(1, int(2 * s)))
    rows = [(title, AMBER)]
    for z in zones:
        c = counts[z.name]
        detail = ", ".join(f"{v} {k}" for k, v in sorted(c.items(), key=lambda kv: -kv[1]))
        rows.append(f"{z.name:<16s}{sum(c.values()):>4d}   {detail}")
    rows.append(f"{'outside areas':<16s}{outside:>4d}")
    mode = f"({tiles[0]}x{tiles[1]} tiles)" if tiles else "(single pass)"
    rows.append((f"{'all objects':<16s}{len(dets):>4d}   {mode}", (200, 200, 200)))
    draw_panel(img, rows, (int(12 * s), int(12 * s)), s * 0.9)
    cv2.imwrite(out_path, img)
    return counts, outside, dets


HIGHWAY_LANES = [
    dict(name="Lane 1 (to cam)", pts=[(0.0, 1.0), (0.15, 1.0), (0.39, 0.40), (0.33, 0.40)]),
    dict(name="Lane 2 (to cam)", pts=[(0.15, 1.0), (0.32, 1.0), (0.45, 0.40), (0.39, 0.40)]),
    dict(name="Lane 3 (to cam)", pts=[(0.32, 1.0), (0.48, 1.0), (0.50, 0.40), (0.45, 0.40)]),
    dict(name="Lane 4 (away)", pts=[(0.55, 1.0), (0.70, 1.0), (0.58, 0.40), (0.52, 0.40)]),
    dict(name="Lane 5 (away)", pts=[(0.70, 1.0), (0.86, 1.0), (0.64, 0.40), (0.58, 0.40)]),
    dict(name="Lane 6 (away)", pts=[(0.86, 1.0), (1.0, 1.0), (1.0, 0.72), (0.70, 0.40), (0.64, 0.40)]),
]


STREET_ROAD = [dict(name="Road", pts=[(0.0, 0.12), (1.0, 0.12), (1.0, 1.0), (0.0, 1.0)])]


det_veh = Detector("models/yolo26n_960.onnx", 960, conf=0.3, classes=["car", "truck", "bus", "motorcycle"])
det_mix = Detector("models/yolo26n_960.onnx", 960, conf=0.3, classes=["car", "truck", "bus", "motorcycle", "bicycle", "person"])
c, outside, _ = count_image("assets/highway_lanes.jpg", O("01b_highway_lanes_still.jpg"), HIGHWAY_LANES, det_veh,
                            tiles=(2, 3), title="VEHICLES PER LANE", label_pos="bottom")
print("per lane:", {k: sum(v.values()) for k, v in c.items()}, "| beyond the lane quads:", outside)
for tag, tiles in (("single", None), ("tiled", (2, 3))):
    c, outside, _ = count_image("assets/street_mixed_traffic.jpg", O(f"02b_street_mix_still_{tag}.jpg"), STREET_ROAD,
                                det_mix, tiles=tiles, merge=True, title="VEHICLE MIX")
    print(f"street {tag}:", c["Road"])
fig, ax = plt.subplots(1, 2, figsize=(18, 5.6))
for a, p in zip(ax, ["01b_highway_lanes_still.jpg", "02b_street_mix_still_tiled.jpg"]):
    a.imshow(cv2.cvtColor(cv2.imread(O(p)), cv2.COLOR_BGR2RGB)); a.axis("off")
plt.tight_layout(); plt.show()

## 6. Use case 2: Parking monitoring

A parked car and a car looking for a space look the same to a detector. Time tells them apart: we track every car and
measure how far it moved over the last second **in the reference frame** (after removing camera drift). Slow means parked;
fast means moving. Only parked cars count towards an area's occupancy. If you know an area's capacity, add
`capacity` and the overlay shows free spaces too.

In [ ]:
def monitor_parking(cfg, detector=None):
    """Count parked cars per parking area and tell them apart from moving cars.

    cfg keys
        video, out, model, imgsz, conf, max_seconds, every
        areas          list of {name, pts (normalized, on the first frame), capacity (optional)}
        align          True to compensate camera drift with ORB + homography
        park_speed     max speed (fraction of frame width per second) to call a car parked
        chart          optional path for an occupancy-over-time PNG
    """
    det = detector or Detector(cfg["model"], cfg.get("imgsz", 640), conf=cfg.get("conf", 0.25),
                               classes=cfg.get("classes", ["car", "truck", "bus"]))
    tracker = IoUTracker(iou_thr=0.2, max_missed=20, min_hits=3, trail_len=30)
    job = VideoJob(cfg["video"], cfg["out"], cfg.get("max_seconds"), cfg.get("out_width"), cfg.get("every", 1))
    W, H = job.out_w, job.out_h
    s = ui_scale(np.zeros((H, W), np.uint8))
    areas = [dict(a, zone=Zone(scaled(a["pts"], W, H), a["name"])) for a in cfg["areas"]]
    fps = job.fps_in / job.every
    hist = {}                                       # track id -> deque of reference-frame ground points
    window = max(3, int(fps))                       # one second of history
    speed_thr = cfg.get("park_speed", 0.012) * W     # pixels per second in the reference frame
    timeline = []
    state = {"aligner": None}
    moving_state = {}                               # track id -> currently moving?
    area_hist = {}                                  # area name -> recent raw counts

    def process(frame, idx):
        dets = det(frame)
        if state["aligner"] is None and cfg.get("align", True):
            state["aligner"] = CameraAligner(frame, motion="homography", boxes=dets)   # a parking lot is a plane
        Hc = state["aligner"].update(frame, dets) if state["aligner"] else np.eye(3)
        Hinv = np.linalg.inv(Hc)
        tracker.update(dets)
        # A parked car that misses a detection for a few frames is still there, so occupancy
        # uses every confirmed track, including those coasting through a short dropout.
        tracks = [t for t in tracker.tracks if t.hits >= tracker.min_hits]

        # classify each car as parked or moving using its motion in the reference frame
        parked, moving = [], []
        edge = 0.015 * W
        for t in tracks:
            g = to_ref(Hc, [t.foot - [0, (t.box[3] - t.box[1]) * 0.25]])[0]   # a point on the car's footprint
            q = hist.setdefault(t.tid, deque(maxlen=window))
            # Only real, uncut observations feed the motion history: a coasting track is a guess,
            # and a box clipped by the frame border grows as the car slides into view, which
            # looks like motion when the camera pans.
            cut = t.box[0] < edge or t.box[2] > W - edge or t.box[1] < edge or t.box[3] > H - edge
            if t.missed == 0 and not cut:
                q.append(g)
            # Judge motion only with ~1 s of history, and compare the median position of the
            # first and last quarter of that window. Two-point speeds over a few frames turn
            # a 3 px box jitter into "45 px/s" and flag parked cars as moving.
            if len(q) >= int(0.8 * window):
                pts = np.array(q)
                k = max(2, len(pts) // 4)
                speed = np.linalg.norm(np.median(pts[-k:], 0) - np.median(pts[:k], 0)) * fps / (len(pts) - k)
                was = moving_state.get(t.tid, False)
                moving_state[t.tid] = speed > (0.5 * speed_thr if was else speed_thr)   # hysteresis
            (moving if moving_state.get(t.tid, False) else parked).append((t, g))

        counts = {}
        for a in areas:
            pts_now = to_ref(Hinv, a["zone"].pts).astype(np.int32)   # area drawn where it is in this frame
            if visible_fraction(pts_now, W, H) < 0.5:                # the camera has panned away from it
                counts[a["name"]] = None
                continue
            raw = sum(1 for _, g in parked if a["zone"].contains(g))
            recent = area_hist.setdefault(a["name"], deque(maxlen=max(3, int(fps // 2))))
            recent.append(raw)
            n = int(round(np.median(recent)))                     # half-second median for a steady readout
            counts[a["name"]] = n
            cap = a.get("capacity")
            full = cap is not None and n >= cap
            col = RED if full else (GREEN if cap is None or n < 0.8 * cap else AMBER)
            z = Zone(pts_now, f"{a['name']}: {n}" + (f"/{cap}" if cap else " parked"), col)
            draw_zone(frame, z, alpha=0.18, scale=s * 0.9)
        timeline.append((idx / job.fps_in, dict(counts), len(moving)))

        for t, _ in parked:
            draw_box(frame, t.box, GREEN, None, s * 0.7, thickness=max(1, int((2 if t.missed == 0 else 1) * s)))
        for t, _ in moving:
            draw_trail(frame, t.trail, AMBER, s)
            draw_box(frame, t.box, AMBER, f"moving #{t.tid}", s * 0.8, thickness=max(2, int(3 * s)))

        rows = [("PARKING MONITOR", AMBER)]
        rows += [f"{k:<12s}{v:>4d}" if v is not None else (f"{k:<12s} out of view", (150, 150, 150))
                 for k, v in counts.items()]
        rows += [(f"parked     {sum(v for v in counts.values() if v):>5d}", GREEN), (f"moving     {len(moving):>5d}", AMBER)]
        if state["aligner"]:
            dx, dy = to_ref(Hinv, [[W / 2, H / 2]])[0] - [W / 2, H / 2]
            rows.append((f"drift {dx:+5.0f},{dy:+5.0f} px", (170, 170, 170)))
        rows.append((f"{job.fps:5.1f} FPS  OpenCV 5", (160, 160, 160)))
        draw_panel(frame, rows, (int(12 * s), int(12 * s)), s * 0.9)
        return frame

    job.run(process)
    if cfg.get("chart"):
        _occupancy_chart(timeline, cfg["chart"])
    return {"timeline": timeline, "fps": job.fps, "frames": len(job.proc_times)}


def visible_fraction(pts, W, H, step=8):
    """Share of a polygon's area that is inside the frame (coarse raster test)."""
    area = cv2.contourArea(pts.astype(np.float32))
    if area < 1:
        return 0.0
    mask = np.zeros((H // step + 1, W // step + 1), np.uint8)
    cv2.fillPoly(mask, [(pts // step).astype(np.int32)], 1)
    return min(1.0, float(mask.sum()) * step * step / area)


def _rolling_median(x, k=15):
    x = np.array([np.nan if v is None else v for v in x], float)
    out = np.full_like(x, np.nan)
    for i in range(len(x)):
        w = x[max(0, i - k // 2): i + k // 2 + 1]
        if np.isfinite(x[i]) and np.isfinite(w).any():
            out[i] = np.nanmedian(w)
    return out


def _occupancy_chart(timeline, path):
    import matplotlib.pyplot as plt          # no matplotlib.use("Agg") here: it would switch off inline plots in a notebook
    t = [x[0] for x in timeline]
    names = list(timeline[0][1].keys()) if timeline else []
    fig, ax = plt.subplots(figsize=(9, 3.6), dpi=120)
    # a half-second rolling median removes single-frame detection flicker; gaps = area out of view
    for n in names:
        ax.plot(t, _rolling_median([x[1][n] for x in timeline]), label=n, lw=2)
    ax.plot(t, _rolling_median([x[2] for x in timeline]), label="moving", lw=1.5, ls="--", color="tab:orange")
    ax.set_xlabel("time (s)"); ax.set_ylabel("cars"); ax.set_title("Parked cars per area over time")
    ax.grid(alpha=0.3); ax.legend(ncol=len(names) + 1, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.2))
    fig.tight_layout(); fig.savefig(path); plt.close(fig)

### Example 4: A supermarket car park, filmed from a drifting drone

The drone pans about 400 pixels during the clip. The four areas are drawn once on the first frame and follow the lot.

In [ ]:
cfg = dict(
    kind="parking", video=M("parking_supermarket.mp4"), out=O("04_parking_supermarket.mp4"),
    chart=O("04_parking_occupancy.png"),
    model="yolo26n_960.onnx", imgsz=960, conf=0.25, every=1, align=True, park_speed=0.025,
    areas=[dict(name="West rows", pts=[(0.0, 0.52), (0.20, 0.51), (0.345, 0.60), (0.21, 0.78), (0.0, 0.87)]),
           dict(name="Center", pts=[(0.375, 0.61), (0.52, 0.62), (0.44, 1.0), (0.22, 1.0)]),
           dict(name="Store front", pts=[(0.55, 0.66), (0.73, 0.65), (0.70, 1.0), (0.50, 1.0)]),
           dict(name="East bays", pts=[(0.82, 0.70), (1.0, 0.70), (1.0, 1.0), (0.82, 1.0)])],
)
res = run_example("4 parking", monitor_parking, cfg)

In [ ]:
show_video(O("04_parking_supermarket.mp4"))
display(Image(O("04_parking_occupancy.png")))

### Example 4b: A still photo, single pass vs tiles

On a single image there is no real-time budget, so we can afford tiling. `parking_image` is a thin wrapper around
`count_image` from the traffic section. The distant cars in the service yard are only a few pixels tall at 960 input
size; a 2 x 3 grid of overlapping tiles gives each car far more pixels.

In [ ]:
def parking_image(img_path, out_path, areas, detector, tiles=None):
    """Single-image parking: cars per area on a still photo (a thin wrapper around count_image)."""
    counts, outside, dets = count_image(img_path, out_path, areas, detector, tiles, title="PARKING (still image)")
    return {k: sum(v.values()) for k, v in counts.items()}, len(dets)


ROWS_AREAS = [
    dict(name="Curb row", pts=[(0.47, 0.36), (0.53, 0.36), (0.62, 1.0), (0.41, 1.0)]),
    dict(name="Center rows", pts=[(0.62, 0.38), (0.79, 0.38), (0.85, 0.68), (0.71, 0.68)]),
    dict(name="East rows", pts=[(0.80, 0.37), (1.0, 0.37), (1.0, 1.0), (0.80, 1.0), (0.855, 0.68)]),
    dict(name="Service yard", pts=[(0.0, 0.42), (0.30, 0.30), (0.36, 0.32), (0.12, 0.68), (0.0, 0.68)]),
]


det_parking = Detector("models/yolo26n_960.onnx", 960, conf=0.25, classes=["car", "truck", "bus"])
src = "assets/parking_rows_still.jpg"
c1, n1 = parking_image(src, O("04b_parking_still_single.jpg"), ROWS_AREAS, det_parking)
c2, n2 = parking_image(src, O("04b_parking_still_tiled.jpg"), ROWS_AREAS, det_parking, tiles=(2, 3))
print("single pass:", c1, "total", n1)
print("tiled 2x3:  ", c2, "total", n2)
fig, ax = plt.subplots(1, 2, figsize=(18, 5.4))
for a, p, t in zip(ax, ["04b_parking_still_single.jpg", "04b_parking_still_tiled.jpg"], [f"single pass: {n1} cars", f"2x3 tiles: {n2} cars"]):
    a.imshow(cv2.cvtColor(cv2.imread(O(p)), cv2.COLOR_BGR2RGB)); a.set_title(t); a.axis("off")
plt.tight_layout(); plt.show()

### Example 4c: Where a COCO detector stops working

COCO's cars are photographed from the side and from the front. Seen straight down, a car is a rectangle the model has
never learned, and it finds nothing even at a 0.15 threshold. For overhead parking cameras you need a model trained on
aerial data (for example a fine-tuned YOLO26 on an aerial dataset); the counting code in this notebook stays the same.

In [ ]:
def topdown_limit(det, path_in, path_out):
    """Run the COCO detector on a straight-down view and show what it thinks it sees."""
    img = cv2.imread(path_in)
    s = ui_scale(img)
    dets = det(img, conf=0.15)                      # low threshold on purpose: show everything it finds
    names = {}
    for d in dets:
        name = COCO[int(d[5])]
        names[name] = names.get(name, 0) + 1
        draw_box(img, d[:4], (0, 190, 255), f"{name} {d[4]:.2f}", s * 0.7)
    rows = [("TOP-DOWN VIEW (COCO model)", (0, 190, 255))]
    rows += [f"{k:<12s}{v:>3d}" for k, v in sorted(names.items(), key=lambda kv: -kv[1])] or ["nothing found"]
    draw_panel(img, rows, (int(12 * s), int(12 * s)), s * 0.9)
    cv2.imwrite(path_out, img)
    return names


print(topdown_limit(Detector("models/yolo26n_960.onnx", 960), "assets/parking_topdown_still.jpg",
                    O("04c_parking_topdown_limit.jpg")))
display(Image(O("04c_parking_topdown_limit.jpg"), width=900))

### Examples 4d and 4e: A parking lot at dusk, and a near-overhead drone shot

Two more lots. The dusk photo tests low light: headlights, wet asphalt and dark cars on dark asphalt. The drone shot is
tilted just enough that the cars keep some side view, but each one is tiny: a single pass finds one car in the whole
photo, and a 3 x 4 grid of tiles finds 34, of which 25 stand inside the lot polygon.

In [ ]:
DUSK_AREAS = [
    dict(name="Left block", pts=[(0.03, 0.63), (0.28, 0.63), (0.30, 0.83), (0.0, 0.86)]),
    dict(name="Center block", pts=[(0.30, 0.62), (0.57, 0.63), (0.58, 0.85), (0.32, 0.84)]),
    dict(name="Right block", pts=[(0.58, 0.70), (0.74, 0.70), (0.74, 0.82), (0.58, 0.82)]),
    dict(name="Front row", pts=[(0.0, 0.86), (0.78, 0.84), (0.78, 1.0), (0.0, 1.0)]),
]


AERIAL_AREAS = [dict(name="Parking lot", pts=[(0.49, 0.42), (0.76, 0.33), (0.80, 0.50), (0.72, 0.93), (0.56, 0.93)])]


c, n = parking_image("assets/parking_dusk_still.jpg", O("04d_parking_dusk_still.jpg"), DUSK_AREAS, det_parking, tiles=(2, 3))
print("dusk:", c, "total", n)
for tag, tiles in (("single", None), ("tiled", (3, 4))):
    c, n = parking_image("assets/parking_aerial_angled.jpg", O(f"04e_parking_aerial_{tag}.jpg"), AERIAL_AREAS, det_parking, tiles)
    print(f"aerial {tag}:", c, "total", n)
fig, ax = plt.subplots(1, 2, figsize=(18, 5.4))
for a, p in zip(ax, ["04d_parking_dusk_still.jpg", "04e_parking_aerial_tiled.jpg"]):
    a.imshow(cv2.cvtColor(cv2.imread(O(p)), cv2.COLOR_BGR2RGB)); a.axis("off")
plt.tight_layout(); plt.show()

## 7. Use case 3: Safety zones

`zone_alert` handles two kinds of zone:

* **static** zones are polygons fixed in the image: a platform edge, a machine area
* a **dynamic** zone is rebuilt every frame around a moving object, here an elliptical safety bubble around a forklift
  (COCO calls it a `truck`)

A person is inside when their foot point is inside. The alert fires after `dwell_s` seconds, which suppresses someone
clipping a corner for a single frame. Each alert saves a snapshot and a row in the CSV log. A person whose box sits
mostly inside the forklift's box is its operator, not a pedestrian at risk, so they are ignored.

In [ ]:
def zone_alert(cfg, detector=None):
    """Raise an alert when a person stays inside a restricted zone.

    Two kinds of zone:
        static   a polygon fixed in the image (platform edge, machine area)
        dynamic  a safety bubble around a moving object, e.g. a forklift (COCO "truck"),
                 rebuilt every frame from that object's box plus a margin

    A person is "in" a zone when their foot point (bottom-center of the box) is
    inside it, which is where they actually stand. An alert fires after
    `dwell_s` seconds inside, saves a snapshot, and is written to the event log.

    cfg keys
        video, out, model, imgsz, conf, max_seconds, every
        zones          list of {name, pts (normalized)}            (static zones)
        bubble         {cls: "truck", margin: 0.6, name}          (optional dynamic zone)
        dwell_s        seconds inside before the alert fires (0 = immediately)
        snapshots      folder for alert snapshots
        csv            path for the event log
    """
    keep = ["person"] + ([cfg["bubble"]["cls"]] if cfg.get("bubble") else [])
    det = detector or Detector(cfg["model"], cfg.get("imgsz", 640), conf=cfg.get("conf", 0.35), classes=keep)
    tracker = IoUTracker(iou_thr=0.25, max_missed=15, min_hits=3)
    job = VideoJob(cfg["video"], cfg["out"], cfg.get("max_seconds"), cfg.get("out_width"), cfg.get("every", 1))
    W, H = job.out_w, job.out_h
    s = ui_scale(np.zeros((H, W), np.uint8))
    zones = [Zone(scaled(z["pts"], W, H), z["name"], RED) for z in cfg.get("zones", [])]
    dwell_need = cfg.get("dwell_s", 0.5)
    inside_since, alerted, events = {}, set(), []
    snap_dir = cfg.get("snapshots")
    if snap_dir:
        os.makedirs(snap_dir, exist_ok=True)
    stats = {"alerts": 0, "max_in_zone": 0}

    def bubble_zones(tracks):
        b = cfg.get("bubble")
        if not b:
            return []
        out = []
        for t in tracks:
            if COCO[t.cls] != b["cls"]:
                continue
            x1, y1, x2, y2 = t.box
            m = b.get("margin", 0.5) * (x2 - x1)
            # Ground-level ellipse around the vehicle's footprint. On an angled camera the floor is
            # foreshortened: a metre towards or away from the camera covers far fewer pixels than a
            # metre sideways, so the bubble is much flatter than it is wide. A tall ellipse would
            # catch people walking in the aisle *behind* the forklift.
            cx, cy = (x1 + x2) / 2, y2 - (y2 - y1) * 0.1
            ax_, ay_ = (x2 - x1) / 2 + m, (y2 - y1) * 0.15 + m * cfg["bubble"].get("depth_ratio", 0.25)
            pts = cv2.ellipse2Poly((int(cx), int(cy)), (int(ax_), int(ay_)), 0, 0, 360, 10)
            out.append((t, Zone(pts, f"{b.get('name', 'safety bubble')} #{t.tid}", AMBER)))
        return out

    def process(frame, idx):
        now = idx / job.fps_in
        tracks = tracker.update(det(frame))
        vehicles = bubble_zones(tracks)
        people = [t for t in tracks if COCO[t.cls] == "person"]
        all_zones = [(None, z) for z in zones] + vehicles

        # people riding the vehicle (box mostly inside the vehicle box) are the operator, not a hazard
        def is_operator(p):
            for v, _ in vehicles:
                ix = max(0, min(p.box[2], v.box[2]) - max(p.box[0], v.box[0]))
                iy = max(0, min(p.box[3], v.box[3]) - max(p.box[1], v.box[1]))
                if ix * iy > 0.6 * (p.box[2] - p.box[0]) * (p.box[3] - p.box[1]):
                    return True
            return False

        hot = set()
        in_zone_now = 0
        for p in people:
            if is_operator(p):
                draw_box(frame, p.box, (150, 150, 150), "operator", s * 0.75)
                continue
            zname = next((z.name for v, z in all_zones if z.contains(p.foot)), None)
            if zname:
                in_zone_now += 1
                key = (p.tid, zname.split(" #")[0])
                inside_since.setdefault(key, now)
                dwell = now - inside_since[key]
                if dwell >= dwell_need:
                    hot.add(zname)
                    if key not in alerted:
                        alerted.add(key)
                        stats["alerts"] += 1
                        events.append({"time_s": round(now, 2), "track_id": p.tid, "zone": key[1], "event": "ALERT"})
                        if snap_dir:
                            snap = frame.copy()
                            draw_box(snap, p.box, RED, f"ALERT {key[1]}", s)
                            cv2.imwrite(os.path.join(snap_dir, f"alert_{stats['alerts']:02d}_t{now:05.1f}s.jpg"), snap)
                col = RED if dwell >= dwell_need else AMBER
                draw_box(frame, p.box, col, f"person #{p.tid} {dwell:.1f}s", s * 0.85, thickness=max(2, int(3 * s)))
            else:
                for key in [k for k in inside_since if k[0] == p.tid]:
                    events.append({"time_s": round(now, 2), "track_id": p.tid, "zone": key[1],
                                   "event": f"left after {now - inside_since.pop(key):.1f}s"})
                draw_box(frame, p.box, GREEN, f"person #{p.tid}", s * 0.85)
            cv2.circle(frame, tuple(p.foot.astype(int)), max(3, int(4 * s)), WHITE, -1, cv2.LINE_AA)
        stats["max_in_zone"] = max(stats["max_in_zone"], in_zone_now)

        # draw zones under a pulse when they are hot
        pulse = 0.25 + 0.2 * (np.sin(idx * 0.6) + 1) / 2
        for v, z in all_zones:
            draw_zone(frame, z, alpha=pulse if z.name in hot else 0.15, color=RED if z.name in hot else z.color, scale=s * 0.9)
            if v is not None:
                draw_box(frame, v.box, AMBER, f"{COCO[v.cls]} #{v.tid}", s * 0.8)
        status = ("ALERT: person in zone", RED) if hot else ("zone clear", GREEN)
        rows = [("SAFETY ZONES", AMBER), status, f"in zone now {in_zone_now:>3d}", f"alerts      {stats['alerts']:>3d}",
                (f"{job.fps:5.1f} FPS  OpenCV 5", (160, 160, 160))]
        draw_panel(frame, rows, (int(12 * s), int(12 * s)), s * 0.9)
        if hot:
            cv2.rectangle(frame, (0, 0), (W - 1, H - 1), RED, max(4, int(8 * s)))
        return frame

    job.run(process)
    if cfg.get("csv"):
        _write_csv(cfg["csv"], events, ["time_s", "track_id", "zone", "event"])
    return {"events": events, "stats": stats, "fps": job.fps, "frames": len(job.proc_times)}

### Example 5: Platform-edge alert at a rail station

The two strips between the yellow safety lines and the tracks are danger zones. Anyone standing there for half a second
triggers an alert.

In [ ]:
cfg = dict(
    kind="zone", video=M("rail_platform.mp4"), out=O("05_zone_platform.mp4"), csv=O("05_zone_platform.csv"),
    snapshots=O("05_platform_alerts"), model="yolo26n_960.onnx", imgsz=960, conf=0.35, every=1, dwell_s=0.5,
    max_seconds=20,
    zones=[dict(name="Platform edge (west)", pts=[(0.415, 0.45), (0.445, 0.45), (0.29, 1.0), (0.215, 1.0)]),
           dict(name="Platform edge (east)", pts=[(0.565, 0.45), (0.59, 0.45), (0.77, 1.0), (0.70, 1.0)])],
)
res = run_example("5 platform", zone_alert, cfg)
print(res["stats"]); print(res["events"][:8])

In [ ]:
show_video(O("05_zone_platform.mp4"))
snaps = sorted(glob.glob(O("05_platform_alerts/*.jpg")))[:2]
for s in snaps: display(Image(s, width=640))

### Example 6: A machine area on a factory floor

The red tape on the floor marks the restricted area. The alert needs one second inside, and every alert is written to
the event log together with how long the person stayed.

In [ ]:
cfg = dict(
    kind="zone", video=M("worker_zone.mp4"), out=O("06_zone_factory.mp4"), csv=O("06_zone_factory.csv"),
    snapshots=O("06_factory_alerts"), model="yolo26n_640.onnx", imgsz=640, conf=0.4, every=2, out_width=1280,
    max_seconds=40,
    dwell_s=1.0,
    zones=[dict(name="Machine area", pts=[(0.36, 0.73), (0.62, 0.565), (0.93, 0.78), (0.80, 1.0), (0.46, 1.0)])],
)
res = run_example("6 factory", zone_alert, cfg)
print(res["stats"])

In [ ]:
show_video(O("06_zone_factory.mp4"))
pd.read_csv(O("06_zone_factory.csv"))

### Example 7: A moving safety bubble around a forklift

No polygon to draw here: the zone is an ellipse at ground level around the forklift, rebuilt from its box every frame.
The driver is recognized as the operator. Anyone else whose feet enter the bubble raises an alert immediately.

The ellipse is much flatter than it is wide, on purpose. On an angled camera the floor is foreshortened: a metre towards
or away from the camera spans far fewer pixels than a metre sideways. Our first version used a taller ellipse and fired
on a worker walking in the aisle *behind* the forklift, metres away in reality but inside the bubble in the image.
`depth_ratio` controls how deep the bubble reaches; a 2D image has no depth, so this is a tuning knob, not a measurement.
In this clip nobody walks up to the forklift, so the correct result is zero alerts.

In [ ]:
cfg = dict(
    kind="zone", video=M("warehouse_forklift.mp4"), out=O("07_zone_forklift.mp4"), csv=O("07_zone_forklift.csv"),
    snapshots=O("07_forklift_alerts"), model="yolo26n_640.onnx", imgsz=640, conf=0.35, every=1, dwell_s=0.0,
    bubble=dict(cls="truck", margin=0.6, name="forklift bubble"),
)
res = run_example("7 forklift", zone_alert, cfg)
print(res["stats"]); print(res["events"][:8])

In [ ]:
show_video(O("07_zone_forklift.mp4"))
show_frames(O("07_zone_forklift.mp4"), [3, 12, 20], cols=3)

### Examples 5b and 6b: Zones on photos

`zone_image` is the single-frame version of the alert: people whose foot point falls inside a polygon are flagged. On the
station photo the zone is the strip between the yellow tactile line and the platform edge; on the warehouse photo it is
the forklift aisle between the racks.

In [ ]:
def zone_image(img_path, out_path, zones_cfg, detector, tiles=None, title="SAFETY ZONES (still image)"):
    """People standing inside restricted polygons on a single photo. Returns ({zone: n}, people)."""
    img = cv2.imread(img_path)
    H, W = img.shape[:2]
    s = ui_scale(img)
    dets = detector.tiled(img, tiles) if tiles else detector(img)
    people = [d for d in dets if COCO[int(d[5])] == "person"]
    zones = [Zone(scaled(z["pts"], W, H), z["name"], RED) for z in zones_cfg]
    inside = {z.name: 0 for z in zones}
    where = []
    for p in people:
        foot = anchor_point(p, "foot")
        z = next((z for z in zones if z.contains(foot)), None)
        where.append((p, foot, z))
        if z is not None:
            inside[z.name] += 1
    for z in zones:
        hot = inside[z.name] > 0
        draw_zone(img, Zone(z.pts, f"{z.name}: {inside[z.name]} inside", RED if hot else AMBER),
                  alpha=0.3 if hot else 0.15, scale=s * 0.9)
    for p, foot, z in where:
        draw_box(img, p[:4], RED if z else GREEN, "ALERT" if z else None, s * 0.8,
                 thickness=max(2, int((3 if z else 2) * s)))
        cv2.circle(img, (int(foot[0]), int(foot[1])), max(3, int(4 * s)), WHITE, -1, cv2.LINE_AA)
    rows = [(title, AMBER)] + [(f"{k:<24s}{v:>3d}", RED if v else GREEN) for k, v in inside.items()]
    rows.append((f"{'people in photo':<24s}{len(people):>3d}", (200, 200, 200)))
    draw_panel(img, rows, (int(12 * s), int(12 * s)), s * 0.9)
    cv2.imwrite(out_path, img)
    return inside, len(people)


PLATFORM_ZONES = [dict(name="Behind the yellow line", pts=[(0.51, 0.44), (0.61, 0.44), (0.61, 1.0), (0.45, 1.0)])]


AISLE_ZONES = [dict(name="Forklift aisle", pts=[(0.44, 0.42), (0.56, 0.42), (0.78, 1.0), (0.22, 1.0)])]


det_people = Detector("models/yolo26n_960.onnx", 960, conf=0.3, classes=["person"])
z, n = zone_image("assets/platform_yellow_line.jpg", O("05b_platform_still.jpg"), PLATFORM_ZONES, det_people, tiles=(2, 2))
print("platform:", z, "of", n, "people")
z, n = zone_image("assets/warehouse_aisle.jpg", O("06b_warehouse_aisle_still.jpg"), AISLE_ZONES, det_people)
print("aisle:", z, "of", n, "people")
fig, ax = plt.subplots(1, 2, figsize=(18, 6.4))
for a, p in zip(ax, ["05b_platform_still.jpg", "06b_warehouse_aisle_still.jpg"]):
    a.imshow(cv2.cvtColor(cv2.imread(O(p)), cv2.COLOR_BGR2RGB)); a.axis("off")
plt.tight_layout(); plt.show()

## 8. Speed on this CPU

First the raw model: one forward pass per engine and input size, on a real 720p frame, after a warm-up. Then the
end-to-end pipeline FPS of every example above (detection + tracking + logic + drawing).

In [ ]:
frame = cv2.imread("assets/parking_rows_still.jpg")
rows = []
for size in (640, 960):
    for engine in ("new", "classic", "auto"):
        d = Detector(f"models/yolo26n_{size}.onnx", size, engine=engine)
        for _ in range(3): d(frame)
        t = time.perf_counter(); n = 20
        for _ in range(n): d(frame)
        ms = (time.perf_counter() - t) / n * 1000
        rows.append((size, engine, ms))
        print(f"yolo26n {size}  ENGINE_{engine.upper():8s} {ms:7.1f} ms  ({1000 / ms:5.1f} FPS)")

print("\nEnd-to-end pipeline FPS per example:")
for k, v in RESULTS.items():
    print(f"  {k:12s} {v['fps']:6.2f} FPS   {v['frames']:5d} frames   {v['model']}")

## 9. Save the results

OpenCV writes `mp4v`, which most browsers will not play. The cell below converts every result to H.264 and zips the
videos, CSV logs, snapshots and charts.

In [ ]:
for p in sorted(glob.glob("outputs/0*.mp4")):
    if "_h264" in p or "_preview" in p:
        continue
    to_h264(p, p.replace(".mp4", "_h264.mp4"), width=1280, crf=23)
!cd outputs && zip -q -r ../opencv5_usecases_outputs.zip . -x "*_preview.mp4"
print(os.path.getsize("opencv5_usecases_outputs.zip") / 1e6, "MB")
# from google.colab import files; files.download("opencv5_usecases_outputs.zip")

## Summary

* A detector answers *what is where*. Tracking, lines and zones turn that into *how many*, *which way* and *who is where
  they should not be*.
* All of it runs on OpenCV 5 and NumPy on a CPU: the new DNN engine for YOLO26, ORB + RANSAC for camera drift,
  `pointPolygonTest` for zones, and a 60-line tracker.
* Pick the input size per scene (640 for near objects, 960 for dense small ones, tiles for still photos), and pick the
  motion model per scene (homography for flat lots, similarity for roads with depth).
* Know the limits: a COCO model does not understand straight-down views; fine-tune on aerial data for that.